In [ ]:
import pandas as pd
import numpy as np
import warnings

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import VotingClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from imblearn.over_sampling import SMOTE

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')


In [ ]:
df_sample_submission = pd.read_csv('sample_submission.csv')
df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')


In [ ]:
TARGET = 'diagnosed_diabetes'
BASE = [col for col in df_train.columns if col not in ['id', TARGET]]
CATS = df_train.select_dtypes('object').columns.to_list()
NUMS = [col for col in BASE if col not in CATS]
print(f'{len(BASE)} Base Features:{BASE}')

## Preprocessing Data
- Split data
- Using SMOTE
- Encoding Data Labeling

In [ ]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

le_encode = {}
for col in CATS:
    if col in train_df.columns:
        le = LabelEncoder()
        all_data = pd.concat([train_df[col], test_df[col]], axis=0).astype(str)
        le.fit(all_data)
        
        train_df[col] = le.transform(train_df[col].astype(str))
        test_df[col] = le.transform(test_df[col].astype(str))
        le_encode[col] = le

X = train_df.drop(columns=[TARGET, 'id'])
Y = train_df[TARGET]

real_test_X = test_df.copy()
if TARGET in real_test_X.columns:
    real_test_X = real_test_X.drop(columns=[TARGET])


X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=y)

print(Y_train.value_counts(normalize=True))


smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, Y_train)


        


In [ ]:
xgb_clf = XGBClassifier(
    n_estimators=1000,    
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=1.2,   
    n_jobs=-1,
    eval_metric='logloss'  
)

lgbm_clf= LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.03,
    num_leaves=31,
    class_weight='balanced', 
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

cat_clf = CatBoostClassifier(
    n_estimators=1000,
    learning_rate=0.03,
    depth=6,
    auto_class_weights='Balanced', 
    random_state=42,
    verbose=0,
    allow_writing_files=False
)

## Run Machine Learning
- Multiple Classification model XGB Model
- Ensemble Modeling

In [ ]:
# XGBoost
xgb_clf.fit(X_train_res, y_train_res, eval_set=[(X_val, Y_val)], verbose=False)
xgb_pred = xgb_clf.predict_proba(X_val)[:, 1]
print(f"XGBoost Validation AUC: {roc_auc_score(Y_val, xgb_pred):.4f}")

# LightGBM
lgbm_clf.fit(X_train_res, y_train_res)
lgbm_pred = lgbm_clf.predict_proba(X_val)[:, 1]
print(f"LightGBM Validation AUC: {roc_auc_score(Y_val, lgbm_pred):.4f}")

# CatBoost
cat_clf.fit(X_train_res, y_train_res, eval_set=(X_val, Y_val), early_stopping_rounds=50)
cat_pred = cat_clf.predict_proba(X_val)[:, 1]
print(f"CatBoost Validation AUC: {roc_auc_score(Y_val, cat_pred):.4f}")

In [ ]:
ensemble_weights = [0.2, 0.4, 0.4]

voting_clf = VotingClassifier(
    estimators=[
        ('xgb', xgb_clf),
        ('lgbm', lgbm_clf),
        ('cat', cat_clf)
    ],
    voting='soft',
    weights=ensemble_weights, 
    n_jobs=-1
)

voting_clf.fit(X_train, Y_train)

In [ ]:
val_pred_proba = voting_clf.predict_proba(X_val)[:, 1]
val_pred_label = voting_clf.predict(X_val)

final_auc = roc_auc_score(Y_val, val_pred_proba)
final_acc = accuracy_score(Y_val, val_pred_label)


print(f"\n[Final Ensemble Result]")
print(f"Validation AUC: {final_auc:.5f}")

print(f"Validation Accuracy: {final_acc:.5f}")
print("\nClassification Report:\n")
print(classification_report(Y_val, val_pred_label))

## Finalized

In [ ]:
if 'id' in real_test_X.columns:
    real_test_X = real_test_X.drop(columns=['id'])

if 'diagnosed_diabetes' in real_test_X.columns:
    real_test_X = real_test_X.drop(columns=['diagnosed_diabetes'])

final_test_proba = voting_clf.predict_proba(real_test_X)[:, 1]
final_test_pred = voting_clf.predict(real_test_X)

submission = pd.DataFrame({
    'Prediction': final_test_pred,
    'Probability': final_test_proba
})

if 'id' in test_df.columns:
    submission.insert(0, 'id', test_df['id'])

submission.to_csv('submission_ensemble_final.csv', index=False)
print("saved submission ensemble csv file ")

In [ ]:
df_final = pd.read_csv('submission_ensemble_final.csv')

print(f"whole data : {len(df_final)}")
print(df_final.head(10))
print(df_final['Probability'].describe())